# Stage 03. Feature Analysis


This notebook performs exploratory analysis on the extracted feature matrices before any model training. Every visualisation here informs the design
decisions in the next steps.

## 0. Feature extraction for S02 (for ablation)

Runs windowing + feature extraction for Subject 2 only at each slide value,
for both chest and wrist. Run this section once.

In [ ]:
import gc
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Paths
CHEST_PREPROCESSED = "../data/processed/wesad/chest_preprocessed.pkl"
WRIST_PREPROCESSED = "../data/processed/wesad/wrist_preprocessed.pkl"

CHEST_CONFIG = "../config_files/dataset/wesad_chest_configuration.json"
WRIST_CONFIG = "../config_files/dataset/wesad_wrist_configuration.json"

WINDOWED_OUTPUT_DIR = "../data/processed/wesad/windowed/"

S2_ABLATION_DIR = "../data/processed/wesad/features/window_ablation_s2"
os.makedirs(S2_ABLATION_DIR, exist_ok=True)

SLIDE_VALUES = [0.25, 0.5, 5.0, 20.0, 60.0]
WINDOW_SIZE_SEC = 60.0
ACC_SUBWINDOW_SEC = 5.0
S2_SID = 2.0

print("S2 ablation config:")
print(f"  Chest preprocessed : {CHEST_PREPROCESSED}")
print(f"  Wrist preprocessed : {WRIST_PREPROCESSED}")
print(f"  Output dir         : {S2_ABLATION_DIR}")
print(f"  Slides             : {SLIDE_VALUES}")

In [ ]:
# Load preprocessed data, filter to S2
print("Loading preprocessed chest data...")
df_chest_full = pd.read_pickle(CHEST_PREPROCESSED)
df_chest_s2 = df_chest_full[df_chest_full["sid"] == S2_SID].copy()
print(
    f"  Chest S2: {len(df_chest_s2):,} rows, labels: {df_chest_s2['label'].value_counts().sort_index().to_dict()}"
)
del df_chest_full
gc.collect()

print("Loading preprocessed wrist data...")
df_wrist_full = pd.read_pickle(WRIST_PREPROCESSED)
df_wrist_s2 = df_wrist_full[df_wrist_full["sid"] == S2_SID].copy()
print(
    f"  Wrist S2: {len(df_wrist_s2):,} rows, labels: {df_wrist_s2['label'].value_counts().sort_index().to_dict()}"
)
del df_wrist_full
gc.collect()

print("Full datasets released from memory.")

In [ ]:
from src.ml_pipeline.feature_extraction import run_feature_pipeline
from src.windowing.wesad_windowing import WESADWindowing

# Extract for each slide value × each device
devices = [
    {"name": "chest", "df_s2": df_chest_s2, "config": CHEST_CONFIG, "wrist": False},
    {"name": "wrist", "df_s2": df_wrist_s2, "config": WRIST_CONFIG, "wrist": True},
]

for dev in devices:
    for slide in SLIDE_VALUES:
        tag = f"{dev['name']}_features_w60s_s{slide}s_S2.pkl"
        output_path = os.path.join(S2_ABLATION_DIR, tag)

        # Skip if already extracted
        if os.path.exists(output_path):
            df_check = pd.read_pickle(output_path)
            print(
                f"  {dev['name']} slide {slide:5.2f}s: EXISTS ({len(df_check)} windows) — skipping"
            )
            del df_check
            continue

        print("\n")
        print(f"  {dev['name'].upper()} — slide {slide}s — windowing S2")
        print("\n")

        windower = WESADWindowing(
            data_path=CHEST_PREPROCESSED,  # pass df directly
            config_path=dev["config"],
            output_dir=WINDOWED_OUTPUT_DIR,
            window_size_sec=WINDOW_SIZE_SEC,
            slide_sec=slide,
            acc_subwindow_sec=ACC_SUBWINDOW_SEC,
            wrist=dev["wrist"],
        )

        windows_s2 = windower.segment(df=dev["df_s2"])
        print(f"  Windows: {len(windows_s2)}")

        df_feat = run_feature_pipeline(
            df=dev["df_s2"],
            windows=windows_s2,
            config_path=dev["config"],
            output_path=output_path,
            wrist=dev["wrist"],
            baseline_label=1.0,
            normalize=True,
            impute=True,
            impute_strategy="median",
        )

        print(f"  Saved: {output_path} ({len(df_feat)} windows)")
        del windows_s2, df_feat
        gc.collect()

print("\nS2 ablation extraction complete (chest + wrist).")

## 1. Setup & Data Loading

In [ ]:
from collections import defaultdict
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

# Project paths congig
PROJECT_ROOT = Path("../")
SRC_ROOT = PROJECT_ROOT / "src"
sys.path.insert(0, str(PROJECT_ROOT))

# Feature matrix paths
CHEST_FEATURES = (
    PROJECT_ROOT / "data" / "processed" / "wesad" / "features" / "chest_features.pkl"
)
WRIST_FEATURES = (
    PROJECT_ROOT / "data" / "processed" / "wesad" / "features" / "wrist_features.pkl"
)

# Window-ablation paths  (S2 only — single subject to save compute)
S2_ABLATION_DIR = (
    PROJECT_ROOT / "data" / "processed" / "wesad" / "features" / "window_ablation_s2"
)
SLIDE_VALUES = [0.25, 0.5, 5.0, 20.0, 60.0]

# Config
CHEST_CONFIG = (
    PROJECT_ROOT / "config_files" / "dataset" / "wesad_chest_configuration.json"
)
WRIST_CONFIG = (
    PROJECT_ROOT / "config_files" / "dataset" / "wesad_wrist_configuration.json"
)

META_COLS = ["sid", "label", "window_idx", "is_overlapped"]

print("Paths configured. Update the constants above if needed.")

In [ ]:
# Load both feature sets:
df_chest = pd.read_pickle(CHEST_FEATURES)
chest_feat_cols = [c for c in df_chest.columns if c not in META_COLS]

df_wrist = pd.read_pickle(WRIST_FEATURES)
wrist_feat_cols = [c for c in df_wrist.columns if c not in META_COLS]

for name, df, fcols in [
    ("CHEST", df_chest, chest_feat_cols),
    ("WRIST", df_wrist, wrist_feat_cols),
]:
    print(f"{name}")
    print(f"  Shape    : {df.shape}")
    print(f"  Features : {len(fcols)}")
    print(f"  Subjects : {sorted(df['sid'].unique().astype(int))}")
    print(f"  Windows  : {len(df):,}")
    print()

## 2. Basic Statistics

In [ ]:
# Label distribution (chest + wrist side by side)
label_names = {1.0: "Baseline", 2.0: "Stress", 3.0: "Amusement"}
label_colors = ["#4CAF50", "#F44336", "#2196F3"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (name, df) in enumerate([("Chest", df_chest), ("Wrist", df_wrist)]):
    # Overall
    counts = df["label"].value_counts().sort_index()
    bars = axes[row, 0].bar(
        [label_names.get(lbl, str(lbl)) for lbl in counts.index],
        counts.values,
        color=label_colors,
        edgecolor="white",
    )
    for bar, val in zip(bars, counts.values):
        axes[row, 0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 50,
            f"{val:,}",
            ha="center",
            fontsize=9,
        )
    axes[row, 0].set_title(f"{name} — window count per label")
    axes[row, 0].set_ylabel("Count")

    # Per-subject
    ct = df.groupby(["sid", "label"]).size().unstack(fill_value=0)
    ct.columns = [label_names.get(c, str(c)) for c in ct.columns]
    ct.plot(
        kind="bar",
        stacked=True,
        ax=axes[row, 1],
        color=label_colors,
        edgecolor="white",
        legend=(row == 0),
    )
    axes[row, 1].set_title(f"{name} — windows per subject & label")
    axes[row, 1].set_xlabel("Subject ID")
    axes[row, 1].set_ylabel("Count")
    axes[row, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# NaN audit
for name, df, fcols in [
    ("CHEST", df_chest, chest_feat_cols),
    ("WRIST", df_wrist, wrist_feat_cols),
]:
    nan_counts = df[fcols].isna().sum()
    nan_features = nan_counts[nan_counts > 0].sort_values(ascending=False)

    print(f"\n{name}:")
    if len(nan_features) == 0:
        print("  No NaN values found.")
    else:
        print(f"  Features with NaN: {len(nan_features)} / {len(fcols)}")
        print(
            f"  Total NaN cells: {nan_counts.sum():,} / {df[fcols].size:,} "
            f"({nan_counts.sum() / df[fcols].size:.4%})"
        )
        print("  Top offenders:")
        print(nan_features.head(10).to_string())

In [ ]:
# Feature groups
def group_features(feat_cols):
    groups = defaultdict(list)
    for f in feat_cols:
        if f.startswith("ECG") or f.startswith("HRV"):
            groups["ECG/HRV"].append(f)
        elif f.startswith("EMG"):
            groups["EMG"].append(f)
        elif "EDA" in f or "SCR" in f or "SCL" in f:
            groups["EDA"].append(f)
        elif f.startswith("RSP") or f.startswith("RESP"):
            groups["RESP"].append(f)
        elif "TEMP" in f or "temp" in f:
            groups["TEMP"].append(f)
        elif "acc" in f.lower() or "ACC" in f:
            groups["ACC"].append(f)
        elif f.startswith("BVP") or f.startswith("PPG"):
            groups["BVP/PPG"].append(f)
        else:
            groups["Other"].append(f)
    return dict(groups)


chest_sensor_groups = group_features(chest_feat_cols, "chest")
wrist_sensor_groups = group_features(wrist_feat_cols, "wrist")

for name, groups in [("CHEST", chest_sensor_groups), ("WRIST", wrist_sensor_groups)]:
    print(f"\n{name} feature groups:")
    for group, feats in sorted(groups.items()):
        print(f"  {group:18s}: {len(feats):3d} features")
    total = sum(len(f) for f in groups.values())
    print(f"  {'TOTAL':18s}: {total:3d}")

## 3. Correlation Analysis

We use Spearman rank correlation rather than Pearson because:
1. Physiological features (HRV spectral ratios, EDA decomposition, etc.) have nonlinear relationships
2. RF splits are rank-based. if two features have identical rank ordering, the RF treats them as redundant
3. Spearman captures all monotonic dependencies, so our redundancy count is more accurate

Pearson is shown alongside for reference.

In [ ]:
# Full correlation heatmap (Spearman, chest + wrist side by side)
cmap = sns.diverging_palette(250, 15, s=75, l=40, n=9, center="light", as_cmap=True)

fig, axes = plt.subplots(1, 2, figsize=(28, 12))

for ax, (name, df, fcols) in zip(
    axes, [("Chest", df_chest, chest_feat_cols), ("Wrist", df_wrist, wrist_feat_cols)]
):
    corr = df[fcols].corr(method="spearman")
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr,
        mask=mask,
        cmap=cmap,
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.1,
        cbar_kws={"shrink": 0.5, "label": "Spearman rho"},
        ax=ax,
    )
    ax.set_title(f"{name} — Spearman Correlation", fontsize=13, pad=12)
    ax.tick_params(axis="both", labelsize=4)

plt.tight_layout()
plt.show()

# Summary: Spearman vs Pearson comparison
for name, df, fcols in [
    ("CHEST", df_chest, chest_feat_cols),
    ("WRIST", df_wrist, wrist_feat_cols),
]:
    corr_s = df[fcols].corr(method="spearman")
    corr_p = df[fcols].corr(method="pearson")
    mask = np.triu(np.ones_like(corr_s, dtype=bool), k=1)
    upper_s = corr_s.where(mask)
    upper_p = corr_p.where(mask)
    total_pairs = len(fcols) * (len(fcols) - 1) // 2
    print(f"{name}:")
    print(f"  {'Threshold':<12} {'Spearman':>12} {'Pearson':>12}")
    for thresh in [0.9, 0.8, 0.7]:
        ns = (upper_s.abs() > thresh).sum().sum()
        np_ = (upper_p.abs() > thresh).sum().sum()
        diff = ns - np_
        print(
            f"  |rho| > {thresh}:  {ns:>6} ({ns/total_pairs:>5.1%})   {np_:>6} ({np_/total_pairs:>5.1%})   +{diff} nonlinear"
        )
    print()

These low percentages are not unexpected, as can be interpreted as a consequence of having many heterogeneous sensor modalities in a single matrix. The features in the matrix span ECG/HRV, EMG, EDA, RESP, TEMP, and ACC which are physiologically very different signals, so the dominant in the count cross-modality correlations will be near-zero.
The high-correlation pairs are expected to cluster within sensor groups, but those intra-group pairs are a small fraction of the full matrix. This is exactly why the per-sensor-group heatmaps will be the more meaningful view, as they are expected tp show much higher within-group correlations.
The Spearman > Pearson difference is also reasonable as it confirms there are some nonlinear relationships.

In [ ]:
# Per-sensor-group correlation heatmaps
for dev_name, groups, df in [
    ("Chest", chest_sensor_groups, df_chest),
    ("Wrist", wrist_sensor_groups, df_wrist),
]:
    groups_multi = {g: f for g, f in groups.items() if len(f) > 1}
    n_groups = len(groups_multi)
    if n_groups == 0:
        continue

    ncols = min(3, n_groups)
    nrows = (n_groups + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 6 * nrows))
    if n_groups == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx, (gname, gfeats) in enumerate(sorted(groups_multi.items())):
        sub_corr = df[gfeats].corr(method="spearman")
        mask_g = np.triu(np.ones_like(sub_corr, dtype=bool), k=1)
        sns.heatmap(
            sub_corr,
            mask=mask_g,
            cmap=cmap,
            center=0,
            vmin=-1,
            vmax=1,
            square=True,
            linewidths=0.3,
            cbar_kws={"shrink": 0.5},
            ax=axes[idx],
        )
        axes[idx].set_title(f"{gname} ({len(gfeats)} feats)", fontsize=10)
        axes[idx].tick_params(axis="both", labelsize=5)

    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{dev_name} — Within-Sensor Spearman Correlation", fontsize=14, y=1.01
    )
    plt.tight_layout()
    plt.show()

## 4. Window Size Ablation (S2 Only)

### 4a. Inter-Window Redundancy
Research question: how redundant are consecutive windows at each slide size?

Hypothesis: with a 60 s window and 0.25 s slide, consecutive windows overlap by 59.75 s so they are
near-duplicates. This inflates the apparent sample count without adding new information.
We measure this with adjacent-window cosine similarity and effective sample size.

### 4b. Inter-Feature Correlation 
Hypothesis: Inter-feature correlation (does feature A move with feature B?) should be
roughly stable across slide sizes because it does not depend on sampling density, but on the feature space alone.

In [ ]:
# Load S2 ablation files + compute inter-window & inter-feature metrics
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

ablation_all = {}

for dev_name in ["chest", "wrist"]:
    ablation_all[dev_name] = {}
    for slide in SLIDE_VALUES:
        path = os.path.join(
            S2_ABLATION_DIR, f"{dev_name}_features_w60s_s{slide}s_S2.pkl"
        )

        if not os.path.exists(path):
            print(f"  {dev_name} slide {slide}s: NOT FOUND")
            continue

        df_abl = pd.read_pickle(path)
        abl_feats = [c for c in df_abl.columns if c not in META_COLS]
        X = df_abl[abl_feats].fillna(0).values

        # Inter-window redundancy: cosine similarity between adjacent windowsc
        X_scaled = StandardScaler().fit_transform(X)
        adj_sims = []
        for i in range(len(X_scaled) - 1):
            sim = cosine_similarity(X_scaled[i : i + 1], X_scaled[i + 1 : i + 2])[0, 0]
            adj_sims.append(sim)
        adj_sims = np.array(adj_sims)

        # Effective sample size estimate:
        # If adjacent windows have similarity s, autocorrelation ~ s,
        # ESS ~ n * (1 - s) / (1 + s)  (geometric series approximation)
        mean_sim = adj_sims.mean()
        n_windows = len(df_abl)
        ess = n_windows * (1 - mean_sim) / (1 + mean_sim) if mean_sim < 1 else 1

        # Inter-feature correlation (Spearman)
        corr_abl = df_abl[abl_feats].corr(method="spearman")
        mask_abl = np.triu(np.ones_like(corr_abl, dtype=bool), k=1)
        upper_vals = corr_abl.where(~mask_abl).stack()

        ablation_all[dev_name][slide] = {
            "n_windows": n_windows,
            "n_features": len(abl_feats),
            # Inter-window metrics
            "mean_adj_cosine": mean_sim,
            "median_adj_cosine": np.median(adj_sims),
            "adj_cosine_values": adj_sims,
            "effective_sample_size": ess,
            "ess_ratio": ess / n_windows,  # what fraction of windows are useful
            # Inter-feature metrics (for reference)
            "mean_abs_feat_corr": upper_vals.abs().mean(),
            "high_feat_corr_frac": (upper_vals.abs() > 0.9).mean(),
        }

        print(
            f"  {dev_name} slide {slide:5.2f}s: {n_windows:>6,} windows | "
            f"adj cosine = {mean_sim:.4f} | "
            f"ESS = {ess:.0f} ({ess/n_windows:.1%}) | "
            f"feat |rho| = {upper_vals.abs().mean():.4f}"
        )
        del df_abl

In [ ]:
# Inter-Window Redundancy Analysis
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
dev_colors = {"chest": "#2196F3", "wrist": "#FF9800"}

# 1) Mean adjacent cosine similarity vs slide size
for dev_name, results in ablation_all.items():
    if not results:
        continue
    slides = sorted(results.keys())
    sims = [results[s]["mean_adj_cosine"] for s in slides]
    axes[0].plot(
        slides,
        sims,
        "o-",
        color=dev_colors[dev_name],
        linewidth=2,
        markersize=8,
        label=dev_name.capitalize(),
    )
axes[0].set_xlabel("Slide size (s)")
axes[0].set_ylabel("Mean Adjacent Cosine Similarity")
axes[0].set_title("Inter-Window Redundancy vs Slide Size")
axes[0].set_xscale("log")
axes[0].xaxis.set_major_formatter(ticker.ScalarFormatter())
axes[0].set_ylim(0, 1.05)
axes[0].axhline(
    y=0.95, color="red", linestyle="--", alpha=0.5, label="Near-duplicate threshold"
)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2) Effective sample size vs raw window count
bar_width = 0.35
for i, (dev_name, results) in enumerate(ablation_all.items()):
    if not results:
        continue
    slides = sorted(results.keys())
    x = np.arange(len(slides))
    # Raw count
    raw = [results[s]["n_windows"] for s in slides]
    axes[1].bar(
        x + i * bar_width,
        raw,
        bar_width,
        color=dev_colors[dev_name],
        alpha=0.3,
        edgecolor="white",
    )
    # ESS
    ess = [results[s]["effective_sample_size"] for s in slides]
    axes[1].bar(
        x + i * bar_width,
        ess,
        bar_width,
        color=dev_colors[dev_name],
        alpha=1.0,
        edgecolor="white",
        label=f"{dev_name.capitalize()} ESS (solid) vs raw (faded)",
    )
axes[1].set_xticks(np.arange(len(SLIDE_VALUES)) + bar_width / 2)
axes[1].set_xticklabels([f"{s}s" for s in SLIDE_VALUES])
axes[1].set_ylabel("Window Count")
axes[1].set_title("Effective Sample Size vs Raw Window Count")
axes[1].legend(fontsize=8)

# 3) ESS ratio (what fraction of windows are actually useful)
for dev_name, results in ablation_all.items():
    if not results:
        continue
    slides = sorted(results.keys())
    ratios = [results[s]["ess_ratio"] for s in slides]
    axes[2].plot(
        slides,
        ratios,
        "s-",
        color=dev_colors[dev_name],
        linewidth=2,
        markersize=8,
        label=dev_name.capitalize(),
    )
axes[2].set_xlabel("Slide size (s)")
axes[2].set_ylabel("ESS / Raw Window Count")
axes[2].set_title("Information Efficiency by Slide Size")
axes[2].set_xscale("log")
axes[2].xaxis.set_major_formatter(ticker.ScalarFormatter())
axes[2].set_ylim(0, 1.05)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary table
print(
    f"\n{'Device':<8} {'Slide':<8} {'Windows':<10} {'Adj Cosine':<12} {'ESS':<10} {'ESS Ratio':<10}"
)
print("-" * 60)
for dev_name, results in ablation_all.items():
    for slide in sorted(results.keys()):
        r = results[slide]
        print(
            f"{dev_name:<8} {slide:<8.2f} {r['n_windows']:<10,} "
            f"{r['mean_adj_cosine']:<12.4f} {r['effective_sample_size']:<10.0f} "
            f"{r['ess_ratio']:<10.1%}"
        )

In [ ]:
# Adjacent Cosine Similarity Distribution + Inter-Feature Stability Check
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Distribution of adjacent cosine similarities per slide
for dev_name, results in ablation_all.items():
    if not results:
        continue
    slides = sorted(results.keys())
    box_data = [results[s]["adj_cosine_values"] for s in slides]
    positions = np.arange(len(slides))
    offset = -0.2 if dev_name == "chest" else 0.2
    bp = axes[0].boxplot(
        box_data,
        positions=positions + offset,
        widths=0.35,
        patch_artist=True,
        showfliers=False,
    )
    color = "#2196F3" if dev_name == "chest" else "#FF9800"
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    # Dummy for legend
    axes[0].bar(0, 0, color=color, alpha=0.7, label=dev_name.capitalize())

axes[0].set_xticks(np.arange(len(SLIDE_VALUES)))
axes[0].set_xticklabels([f"{s}s" for s in SLIDE_VALUES])
axes[0].set_xlabel("Slide size")
axes[0].set_ylabel("Adjacent Window Cosine Similarity")
axes[0].set_title("Distribution of Inter-Window Redundancy by Slide Size (S2)")
axes[0].axhline(y=0.95, color="red", linestyle="--", alpha=0.5, label="Near-duplicate")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis="y")

# Bottom: Inter-feature correlation stability. expected flat
for dev_name, results in ablation_all.items():
    if not results:
        continue
    slides = sorted(results.keys())
    feat_corrs = [results[s]["mean_abs_feat_corr"] for s in slides]
    color = "#2196F3" if dev_name == "chest" else "#FF9800"
    axes[1].plot(
        slides,
        feat_corrs,
        "o-",
        color=color,
        linewidth=2,
        markersize=8,
        label=dev_name.capitalize(),
    )

axes[1].set_xlabel("Slide size (s)")
axes[1].set_ylabel("Mean |Spearman rho| (inter-feature)")
axes[1].set_title("Inter-Feature Correlation Stability Check (should be flat)")
axes[1].set_xscale("log")
axes[1].xaxis.set_major_formatter(ticker.ScalarFormatter())
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Interpretation
- At 0.25s slide, adjacent windows share 59.75/60 roughly 99% of their data. 
- The cosine similarity being very close to 1.0 confirms near-duplicate windows.
- At 60s slide (no overlap), consecutive windows are independent so similarity drops. 

- Inter-feature correlation should be roughly constant across slide sizes. This would confirm that slide size affects sample redundancy, not feature geometry. 
- The slight upward trend is also expected, as given larger windows , biosignals with different physiological lags get a bigger time window to co-move and allow us to observe their relationship.

## 5. t-SNE Visualisation

Two views per device:
1. By label: do stress/baseline/amusement form separable clusters?
2. By subject: how much subject drift exists?

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

MAX_TSNE_SAMPLES = 10000
np.random.seed(42)

label_names_map = {1.0: "Baseline", 2.0: "Stress", 3.0: "Amusement"}
label_colors_map = {1.0: "#4CAF50", 2.0: "#F44336", 3.0: "#2196F3"}

fig, axes = plt.subplots(2, 2, figsize=(20, 16))

for row, (name, df, fcols) in enumerate(
    [("Chest", df_chest, chest_feat_cols), ("Wrist", df_wrist, wrist_feat_cols)]
):
    # Subsample
    if len(df) > MAX_TSNE_SAMPLES:
        idx = np.random.choice(len(df), MAX_TSNE_SAMPLES, replace=False)
        df_sub = df.iloc[idx].copy()
    else:
        df_sub = df.copy()

    X = StandardScaler().fit_transform(df_sub[fcols].fillna(0))
    print(f"Running t-SNE for {name} ({len(df_sub)} samples)...")
    emb = TSNE(
        n_components=2, perplexity=30, random_state=42, max_iter=1000, init="pca"
    ).fit_transform(X)
    df_sub["tsne_1"] = emb[:, 0]
    df_sub["tsne_2"] = emb[:, 1]

    # By label
    for lv, ln in label_names_map.items():
        m = df_sub["label"] == lv
        axes[row, 0].scatter(
            df_sub.loc[m, "tsne_1"],
            df_sub.loc[m, "tsne_2"],
            c=label_colors_map[lv],
            label=ln,
            alpha=0.5,
            s=10,
            edgecolors="none",
        )
    axes[row, 0].set_title(f"{name} — t-SNE by Label", fontsize=13)
    axes[row, 0].legend(markerscale=3, fontsize=9)
    axes[row, 0].set_xlabel("t-SNE 1")
    axes[row, 0].set_ylabel("t-SNE 2")

    # By subject
    subjects = sorted(df_sub["sid"].unique())
    cmap_s = plt.cm.get_cmap("tab20", len(subjects))
    for i, sid in enumerate(subjects):
        m = df_sub["sid"] == sid
        axes[row, 1].scatter(
            df_sub.loc[m, "tsne_1"],
            df_sub.loc[m, "tsne_2"],
            c=[cmap_s(i)],
            label=f"S{int(sid)}",
            alpha=0.5,
            s=10,
            edgecolors="none",
        )
    axes[row, 1].set_title(f"{name} — t-SNE by Subject", fontsize=13)
    axes[row, 1].legend(markerscale=3, fontsize=7, ncol=3, loc="upper right")
    axes[row, 1].set_xlabel("t-SNE 1")
    axes[row, 1].set_ylabel("t-SNE 2")

plt.suptitle("t-SNE Embeddings (Chest vs Wrist)", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()
print("t-SNE complete.")

Interpretations: 
- Left plot: There is a very clear separation observed between the 3 different groups. The separable regions that can be observed prove that the features are fairly discriminative and the label quality is good. This is a very positive sign for the downstream classifier.
- The baseline class (green) forms a large, dense, centrally-located mass. This is expected because baseline is the longest condition and physiologically the most "resting," so windows are more homogeneous and tightly packed together. Stress (red) and Amusement (blue) appear predominantly at the periphery, forming distinct clusters around the baseline core. This arrangement suggests that both stress and amusement represent physiological departures from baseline, which aligns perfectly with the arousal theory underlying WESAD.
- The separation is especially clean for chest, while for wrist there is a bit more fuzziness as those sensors are susceptible to noise artifacts. This is consistent with existing literature. 
- Right plot: This plot reveals subject-specific structure. Several subjects form their own clusters, which means inter-subject variability is large relative to condition variability. 
Despite the subject-level clustering though, if we cross reference both plots we can observe that within each subject's cluster, the condition colors still show some internal structure. This means condition signal exists even within individual subjects, supporting the validity of LOSO-CV and subject-independent evaluation since having windows from the same subject in both train and test sets would constitute data leakage.

## 6. Class Separability: Fisher Discriminant Ratio

FDR = variance_between / variance_within per feature. Higher fdr means better class separation.
We compute it here for both devices, for binary classification (stress vs non-stress).

In [ ]:
from scipy.stats import f_oneway


def compute_fdr(df, feat_cols, label_map=None):
    """Compute binary FDR and ANOVA p-values for all features."""
    if label_map is None:
        label_map = {1.0: 0, 2.0: 1, 3.0: 0}
    df_tmp = df.copy()
    df_tmp["binary_label"] = df_tmp["label"].map(label_map)

    fdr = {}
    pvals = {}
    for feat in feat_cols:
        vals = df_tmp[feat].dropna()
        labels = df_tmp.loc[vals.index, "binary_label"]
        g0 = vals[labels == 0]
        g1 = vals[labels == 1]
        if len(g0) < 2 or len(g1) < 2:
            continue
        denom = g0.var(ddof=0) + g1.var(ddof=0)
        fdr[feat] = (g1.mean() - g0.mean()) ** 2 / denom if denom > 0 else 0.0
        _, p = f_oneway(g0, g1)
        pvals[feat] = p

    return pd.Series(fdr).sort_values(ascending=False), pd.Series(pvals)


chest_fdr, chest_anova = compute_fdr(df_chest, chest_feat_cols)
wrist_fdr, wrist_anova = compute_fdr(df_wrist, wrist_feat_cols)

for name, fdr_s, anova_s in [
    ("CHEST", chest_fdr, chest_anova),
    ("WRIST", wrist_fdr, wrist_anova),
]:
    print(f"\n{name} — Top 15 features by FDR:")
    print(fdr_s.head(15).to_string())
    print(
        f"  Features with ANOVA p < 0.001: {(anova_s < 0.001).sum()} / {len(anova_s)}"
    )

In [ ]:
# FDR bar charts side by side
top_n = 25

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

for ax, (name, fdr_s) in zip(axes, [("Chest", chest_fdr), ("Wrist", wrist_fdr)]):
    top = fdr_s.head(top_n)
    ax.barh(
        range(top_n),
        top.values[::-1],
        color="#2196F3" if "Chest" in name else "#FF9800",
        edgecolor="white",
    )
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top.index[::-1], fontsize=7)
    ax.set_xlabel("Fisher Discriminant Ratio")
    ax.set_title(f"{name} — Top {top_n} Features (Stress vs Non-Stress)")
    ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

## 7. Feature Distributions: Top Features by Label

Violin plots for the top FDR features per device.

In [ ]:
# Violin plots
label_names_map = {1.0: "Baseline", 2.0: "Stress", 3.0: "Amusement"}

for dev_name, df, fdr_s in [
    ("Chest", df_chest, chest_fdr),
    ("Wrist", df_wrist, wrist_fdr),
]:
    top_feats = fdr_s.head(10).index.tolist()
    n_feats = len(top_feats)
    ncols = 5
    nrows = (n_feats + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(22, 4.5 * nrows))
    axes_flat = axes.flatten() if n_feats > 1 else [axes]

    for i, feat in enumerate(top_feats):
        plot_df = df[["label", feat]].dropna().copy()
        plot_df["Condition"] = plot_df["label"].map(label_names_map)
        sns.violinplot(
            data=plot_df,
            x="Condition",
            y=feat,
            palette={
                "Baseline": "#4CAF50",
                "Stress": "#F44336",
                "Amusement": "#2196F3",
            },
            inner="quartile",
            ax=axes_flat[i],
            cut=0,
            order=["Baseline", "Stress", "Amusement"],
        )
        axes_flat[i].set_title(feat, fontsize=9, fontweight="bold")
        axes_flat[i].set_xlabel("")
        axes_flat[i].tick_params(axis="x", labelsize=8)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle(
        f"{dev_name} — Top {n_feats} FDR Features by Condition", fontsize=14, y=1.01
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-subject KDE for top-3 features (subject drift check)
for dev_name, df, fdr_s in [
    ("Chest", df_chest, chest_fdr),
    ("Wrist", df_wrist, wrist_fdr),
]:
    top_3 = fdr_s.head(3).index.tolist()
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    for i, feat in enumerate(top_3):
        for sid in sorted(df["sid"].unique()):
            sub_vals = df.loc[df["sid"] == sid, feat].dropna()
            if len(sub_vals) > 10:
                sub_vals.plot.kde(
                    ax=axes[i], alpha=0.4, linewidth=1, label=f"S{int(sid)}"
                )
        axes[i].set_title(f"{feat}", fontsize=10)
        axes[i].set_xlabel(feat)
        axes[i].legend(fontsize=6, ncol=2)

    plt.suptitle(
        f"{dev_name} — Subject Variability (Top 3 FDR Features)", fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.show()

We observe that subjects have quite notably different KDE shapes for the same feature, which indicates that the feature's distribution varies significantly by person. This visually confirms why per-subject normalization and LOSO-CV are necessary.

## 8. Feature Importance Preview (Quick RF)

Exploratory RF on all data (no LOSO). Compares MDI importance to FDR ranking.

Note: because MDI favours high-cardinality features (more unique values meaning more possible split thresholds for the decision trees therefore more opportunities for the feature to be useful), this is why we are also cross-validating MDI rankings against the FDR with Spearman rank correlation (ρ), which is useful in this case as it measures monotonic agreement between two rankings without assuming linearity. The hypothesis being tested is that features ranked highly by FDR also tend to be ranked highly by MDI. A high ρ here gives confidence the feature importance signal is robust across methods.

In [ ]:
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier

fig_scatter, axes_scatter = plt.subplots(1, 2, figsize=(16, 7))
fig_bars, axes_bars = plt.subplots(1, 2, figsize=(22, 9))

for idx, (dev_name, df, fcols, fdr_s) in enumerate(
    [
        ("Chest", df_chest, chest_feat_cols, chest_fdr),
        ("Wrist", df_wrist, wrist_feat_cols, wrist_fdr),
    ]
):
    X = df[fcols].fillna(0).values
    y = df["label"].map({1.0: 0, 2.0: 1, 3.0: 0}).values

    print(f"Training quick RF for {dev_name}...")
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X, y)
    imp = pd.Series(rf.feature_importances_, index=fcols).sort_values(ascending=False)

    # Bar chart
    top_n = 25
    top_imp = imp.head(top_n)
    axes_bars[idx].barh(
        range(top_n),
        top_imp.values[::-1],
        color="#FF9800" if idx else "#2196F3",
        edgecolor="white",
    )
    axes_bars[idx].set_yticks(range(top_n))
    axes_bars[idx].set_yticklabels(top_imp.index[::-1], fontsize=7)
    axes_bars[idx].set_xlabel("MDI Importance")
    axes_bars[idx].set_title(f"{dev_name} — Top {top_n} RF Importances")
    axes_bars[idx].grid(True, alpha=0.3, axis="x")

    # FDR vs RF rank correlation
    common = fdr_s.index.intersection(imp.index)
    fdr_ranks = fdr_s[common].rank(ascending=False)
    imp_ranks = imp[common].rank(ascending=False)
    rho, p_val = spearmanr(fdr_ranks, imp_ranks)

    axes_scatter[idx].scatter(fdr_ranks, imp_ranks, alpha=0.4, s=15, color="#673AB7")
    axes_scatter[idx].plot([0, len(common)], [0, len(common)], "k--", alpha=0.3)
    axes_scatter[idx].set_xlabel("FDR Rank")
    axes_scatter[idx].set_ylabel("RF Importance Rank")
    axes_scatter[idx].set_title(
        f"{dev_name} — FDR vs RF (rho={rho:.3f}, p={p_val:.2e})"
    )
    axes_scatter[idx].set_aspect("equal")

    print(f"  {dev_name} Spearman rho = {rho:.4f}")

fig_bars.suptitle("RF Feature Importances (Exploratory)", fontsize=14, y=1.01)
fig_bars.tight_layout()
fig_bars.show()

fig_scatter.suptitle("FDR vs RF Importance Rank Agreement", fontsize=14, y=1.01)
fig_scatter.tight_layout()
fig_scatter.show()

Note: because the quick RF has "seen" the entire set including test subjects we expect the importances to be inflated. Therefore this process is only used for visualization purposes, with acknowledgment of this limitation

## 9. Feature Selection Method Comparison

Tree FS, KBest, and PCA fitted on the full matrix (preview only-> actual FS is per-fold in trainiung pipeline).

In [ ]:
from src.ml_pipeline.feature_selection import (
    KBestSelector,
    PCASelector,
    TreeFeatureSelector,
    VarianceSelector,
)


def run_fs_comparison(df, feat_cols, device_name):
    """Run all three FS methods and return overlap stats."""
    y = df["label"].map({1.0: 0, 2.0: 1, 3.0: 0}).astype(int)

    # Variance threshold
    var_sel = VarianceSelector(threshold=0.0)
    df_var = var_sel.fit_transform(df)
    var_feats = [c for c in df_var.columns if c not in META_COLS]
    print(f"  {device_name} variance threshold: {len(feat_cols)} -> {len(var_feats)}")

    # Tree FS
    tree_sel = TreeFeatureSelector(n_estimators=200, threshold="mean", random_state=42)
    tree_sel.fit(df_var, y)
    tree_feats = set(tree_sel.selected_features)

    # KBest (match k to tree count)
    k_val = len(tree_feats)
    kbest_sel = KBestSelector(k=k_val)
    kbest_sel.fit(df_var, y)
    kbest_feats = set(kbest_sel.selected_features)

    # PCA
    pca_sel = PCASelector(n_components=0.95)
    pca_sel.fit(df_var)
    pca_n = len(pca_sel.selected_features)

    overlap = tree_feats & kbest_feats
    jaccard = (
        len(overlap) / len(tree_feats | kbest_feats) if tree_feats | kbest_feats else 0
    )

    print(
        f"  Tree: {len(tree_feats)}, KBest(k={k_val}): {len(kbest_feats)}, PCA: {pca_n} components"
    )
    print(f"  Overlap: {len(overlap)}, Jaccard: {jaccard:.2%}")

    return {
        "tree": tree_feats,
        "kbest": kbest_feats,
        "pca_n": pca_n,
        "overlap": overlap,
        "jaccard": jaccard,
        "k_val": k_val,
        "tree_sel": tree_sel,
        "kbest_sel": kbest_sel,
    }


print("CHEST:")
chest_fs = run_fs_comparison(df_chest, chest_feat_cols, "Chest")
print("\nWRIST:")
wrist_fs = run_fs_comparison(df_wrist, wrist_feat_cols, "Wrist")

We observe that both devices show moderately good agreement between Tree FS and KBest. Out of every feature selected by either method, roughly 57-58% were selected by both. The methods are coming from fundamentally different mathematical perspectives so this level of agreement is a scientifically significant one. As Jaccard similarity measures the overlap between two sets, this finding indicates that the features chosen have genuine importance and meaning.

In [ ]:
# Overlap bar charts
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (name, fs) in zip(axes, [("Chest", chest_fs), ("Wrist", wrist_fs)]):
    tree_only = fs["tree"] - fs["kbest"]
    kbest_only = fs["kbest"] - fs["tree"]
    overlap = fs["overlap"]

    cats = ["Both agree", "Tree FS only", "KBest only"]
    vals = [len(overlap), len(tree_only), len(kbest_only)]
    colors = ["#4CAF50", "#FF9800", "#2196F3"]

    bars = ax.barh(cats, vals, color=colors, edgecolor="white", height=0.5)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f"{val}",
            va="center",
            fontsize=11,
        )
    ax.set_xlabel("Number of features")
    ax.set_title(
        f"{name} — Tree vs KBest (k={fs['k_val']}, Jaccard={fs['jaccard']:.0%})"
    )
    ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

In [ ]:
# Importance vs F-score scatter
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (name, fs) in zip(axes, [("Chest", chest_fs), ("Wrist", wrist_fs)]):
    tree_imp = fs["tree_sel"].importances
    kbest_scores = fs["kbest_sel"].scores

    # Normalise to [0, 1]
    tn = (tree_imp - tree_imp.min()) / (tree_imp.max() - tree_imp.min())
    kn = (kbest_scores - kbest_scores.min()) / (kbest_scores.max() - kbest_scores.min())

    common = tn.index.intersection(kn.index)
    ax.scatter(tn[common], kn[common], alpha=0.4, s=15, c="#9C27B0", zorder=1)

    for feat in common:
        in_t = feat in fs["tree"]
        in_k = feat in fs["kbest"]
        if in_t and in_k:
            ax.scatter(tn[feat], kn[feat], c="#4CAF50", s=40, zorder=5, marker="o")
        elif in_t:
            ax.scatter(tn[feat], kn[feat], c="#FF9800", s=40, zorder=5, marker="^")
        elif in_k:
            ax.scatter(tn[feat], kn[feat], c="#2196F3", s=40, zorder=5, marker="s")

    ax.set_xlabel("Tree FS importance (norm)")
    ax.set_ylabel("KBest F-score (norm)")
    ax.set_title(f"{name} Feature Scoring Comparison")
    ax.grid(True, alpha=0.3)

    # Explicit legend handles
    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="#9C27B0",
            markersize=7,
            alpha=0.6,
            label="Not selected",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="#4CAF50",
            markersize=8,
            label="Both selected",
        ),
        Line2D(
            [0],
            [0],
            marker="^",
            color="w",
            markerfacecolor="#FF9800",
            markersize=8,
            label="Tree FS only",
        ),
        Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor="#2196F3",
            markersize=8,
            label="KBest only",
        ),
    ]
    ax.legend(handles=legend_handles, fontsize=8, loc="upper left")

plt.tight_layout()
plt.show()